In [3]:
!pip install -q huggingface_hub datasets sacrebleu rouge_score seaborn matplotlib

In [4]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
GITHUB_TOKEN = secrets.get_secret("GITHUB_TOKEN")
HF_TOKEN = secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

REPO_URL = f"https://{GITHUB_TOKEN}@github.com/CL3-410/individual-project-ManyaJain111.git"

if not os.path.isdir("repo"):
    !git clone -b phase2 {REPO_URL} repo

Cloning into 'repo'...
remote: Enumerating objects: 351, done.
remote: Counting objects: 100% (351/351), done.
remote: Compressing objects: 100% (275/275), done.
remote: Total 351 (delta 161), reused 205 (delta 65), pack-reused 0 (from 0)
Receiving objects: 100% (351/351), 19.50 MiB | 22.34 MiB/s, done.
Resolving deltas: 100% (161/161), done.


In [8]:
import os, glob, re

# Find the exact path of the final checkpoint
ckpt_paths = glob.glob("/kaggle/input/datasets/manyaj123456/prevop/**/model_final.pt", recursive=True)
if not ckpt_paths:
    raise FileNotFoundError("Could not find model_final.pt in the mounted dataset!")
MODEL_PATH = ckpt_paths[0]
print(f"Found checkpoint at: {MODEL_PATH}")
os.environ["MODEL_PATH"] = MODEL_PATH # Save for bash command in Cell 4

# Read evaluate.py
eval_path = "repo/phase2/src/evaluate.py"
with open(eval_path, "r", encoding="utf-8") as f:
    src = f.read()

# 1. Add the class to the very top of the file to avoid indentation conflicts
if "class WhitespaceTokenizer" not in src:
    tokenizer_class = "class WhitespaceTokenizer:\n    def tokenize(self, text):\n        return text.split()\n\n"
    src = tokenizer_class + src

# 2. Update just the parameters of the RougeScorer safely
src = re.sub(
    r"rouge_scorer\.RougeScorer\(\['rougeL'\], use_stemmer=False.*?\)", 
    "rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False, tokenizer=WhitespaceTokenizer())", 
    src
)

# Save the patched file
with open(eval_path, "w", encoding="utf-8") as f:
    f.write(src)
print("Patched evaluate.py successfully.")

Found checkpoint at: /kaggle/input/datasets/manyaj123456/prevop/kaggle/working/hindi_checkpoints/model_final.pt
Patched evaluate.py successfully.


In [12]:
# 1. Force git to restore the original, unbroken file
!git -C repo restore phase2/src/evaluate.py

import os

# 2. Read the fresh file
eval_path = "repo/phase2/src/evaluate.py"
with open(eval_path, "r", encoding="utf-8") as f:
    src = f.read()

# 3. Add the tokenizer class safely at the very top
src = "class WhitespaceTokenizer:\n    def tokenize(self, text): return text.split()\n\n" + src

# 4. Replace the RougeScorer string exactly
src = src.replace(
    "RougeScorer(['rougeL'], use_stemmer=False)", 
    "RougeScorer(['rougeL'], use_stemmer=False, tokenizer=WhitespaceTokenizer())"
)

# 5. Save the fixed file
with open(eval_path, "w", encoding="utf-8") as f:
    f.write(src)

print("evaluate.py has been restored and cleanly patched!")

evaluate.py has been restored and cleanly patched!


In [14]:
import os, json, re, string, glob, tarfile, numpy as np
from multiprocessing import Pool, cpu_count
from huggingface_hub import hf_hub_download

EXTRACT_DIR = '/kaggle/working/lma_extracted/hindi-lma-corpus'
TOK_PATH    = 'repo/phase2/tokenizers/hindi_tokenizer_16000.json'
OUT_DIR     = '/kaggle/working'
CHUNK       = 5000

# 1. Download and extract dataset
if not os.path.exists(f'{EXTRACT_DIR}/.done'):
    print("[Dataset] Downloading hindi_submission_phase1.tar.gz...")
    tar_path = hf_hub_download(repo_id="manyajain111/hindi-lma-corpus", filename="hindi_submission_phase1.tar.gz", repo_type="dataset")
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall(path=EXTRACT_DIR)
    open(f'{EXTRACT_DIR}/.done', 'w').close()
    print("[Dataset] Extraction done!")

# 2. Tokenizer setup
PUNCT    = re.escape(string.punctuation + "।॥''\"\"")
WORD_PAT = re.compile(rf" ?[^{PUNCT}\s]+| ?[{PUNCT}]+", re.UNICODE)

def init_worker(tok_path):
    global _vocab, _merge_ranks, _b2c, _cache
    with open(tok_path) as f: data = json.load(f)
    _vocab = data['vocab']
    merges = [((a, b), a + b) for a, b in data['merges']]
    _merge_ranks = {p: i for i, (p, _) in enumerate(merges)}
    _b2c = {i: chr(i) for i in range(256)}
    _cache = {}

def encode_chunk(lines):
    out = []
    for line in lines:
        line = line.strip()
        if not line: continue
        for w in WORD_PAT.findall(line):
            if w not in _cache:
                s = [_b2c[b] for b in w.encode('utf-8')]
                while len(s) > 1:
                    pairs  = [(s[i], s[i+1]) for i in range(len(s)-1)]
                    ranked = [(_merge_ranks[p], i) for i, p in enumerate(pairs) if p in _merge_ranks]
                    if not ranked: break
                    _, i = min(ranked)
                    s = s[:i] + [s[i] + s[i+1]] + s[i+2:]
                _cache[w] = [_vocab.get(x, 0) for x in s]
            out.extend(_cache[w])
    return out

# 3. Process test.txt
fpaths = glob.glob(f'{EXTRACT_DIR}/**/test.txt', recursive=True)
if not fpaths:
    raise FileNotFoundError("Could not find test.txt in the extracted dataset.")

print(f'Reading {os.path.basename(fpaths[0])}...')
lines = open(fpaths[0], encoding='utf-8', errors='ignore').readlines()
chunks = [lines[i:i+CHUNK] for i in range(0, len(lines), CHUNK)]

out_path = f'{OUT_DIR}/hindi_test.bin'
if os.path.exists(out_path): os.remove(out_path)

total_tokens = 0
with open(out_path, 'wb') as f_out:
    with Pool(cpu_count(), initializer=init_worker, initargs=(TOK_PATH,)) as pool:
        for r in pool.imap(encode_chunk, chunks):
            arr = np.array(r, dtype=np.uint16)
            f_out.write(arr.tobytes())
            total_tokens += len(arr)

print(f'✅ Saved {total_tokens:,} tokens -> {out_path}')

[Dataset] Downloading hindi_submission_phase1.tar.gz...


hindi_submission_phase1.tar.gz:   0%|          | 0.00/1.51G [00:00<?, ?B/s]

/tmp/ipykernel_58/2508766813.py:16: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=EXTRACT_DIR)


[Dataset] Extraction done!
Reading test.txt...
✅ Saved 28,906,057 tokens -> /kaggle/working/hindi_test.bin


In [15]:
!python repo/phase2/src/evaluate.py \
    --checkpoint {MODEL_PATH} \
    --tokenizer repo/phase2/tokenizers/hindi_tokenizer_16000.json \
    --dataset manyajain111/hindi-lma-corpus \
    --language hindi \
    --output_dir /kaggle/working/hindi_reports

Device: cuda
Loaded model: 24.22M params
[Dataset] Memory-mapping /kaggle/working/hindi_test.bin...
[Dataset] 28,906,057 tokens ready.
Evaluating PPL/BPB: 100%|█████████████████████| 200/200 [00:08<00:00, 24.00it/s]

Test Loss: 3.6663 nats | PPL: 39.11 | BPB: 0.5131 | avg bytes/tok: 10.31
Sampling Temp 1.5: 100%|██████████████████████████| 5/5 [00:01<00:00,  4.78it/s]

Generation Metrics:
  Temp greedy: BLEU=0.95 chrF=12.21 ROUGE-L=0.1243 Dist-1=0.4270 Dist-2=0.6215 rep=0.2659
    Sample generation: मैं आज सुबह से ही शुरू हो गई थी। इस बार भी कई बार लोग अपने घरों में ही कैद हो गए हैं। ऐसे में अब लोग अपने घरों में कैद ह
  Temp 0.5: BLEU=0.69 chrF=12.04 ROUGE-L=0.0835 Dist-1=0.4592 Dist-2=0.7069 rep=0.1467
    Sample generation: मैं आज इस बात की जानकारी मिली है कि इन तीनों शहरों में बिजली की खपत कम है। इन तीनों शहरों में यह खपत कम है। इस वजह से लो
  Temp 1.0: BLEU=0.84 chrF=14.04 ROUGE-L=0.1223 Dist-1=0.7594 Dist-2=0.9731 rep=0.0000
    Sample generation: मैं आज रतकेआवेदनमें।लगता है 436 श

In [7]:
import sys, torch, json
sys.path.insert(0, "repo/phase2/src")
from model import GPTLanguageModel
from tokenizer_wrapper import TokenizerWrapper

cfg = json.load(open("repo/phase2/config/model_h_config.json"))
tok = TokenizerWrapper("repo/phase2/tokenizers/hindi_tokenizer_16000.json")
model = GPTLanguageModel(**cfg)

# MODEL_PATH is carried over from Cell 3
print(f"Loading checkpoint from: {MODEL_PATH}")
ckpt = torch.load(MODEL_PATH, map_location="cpu", weights_only=True)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

ids = tok.encode("मैं आज बाजार गया था।")
x = torch.tensor([ids])
with torch.no_grad():
    logits_orig, _ = model(x)

x2 = x.clone()
x2[0, -1] = (x2[0, -1] + 137) % cfg["vocab_size"]
with torch.no_grad():
    logits_perturbed, _ = model(x2)

diff = (logits_orig[:, :-1] - logits_perturbed[:, :-1]).abs().max().item()
print(f"Max logit diff at earlier positions: {diff:.2e}")
assert diff < 1e-5, "Causal mask is leaking future information!"
print("✅ Causal masking verified: changing the last token does not affect earlier positions' logits.")

Loading checkpoint from: /kaggle/input/datasets/manyaj123456/prevop/kaggle/working/hindi_checkpoints/model_final.pt
Max logit diff at earlier positions: 0.00e+00
✅ Causal masking verified: changing the last token does not affect earlier positions' logits.
